In [32]:
from fastai.torch_basics import *
from fastai.data.all import *
from fastai.text.data import *
from tokenizers import Tokenizer
from datasets import load_dataset

In [33]:
#| default_exp text.tokenizer


# Text tokenizer

In [34]:
EOS_TOK = "<eos>"
BOS_TOK = "<bos>"
PAD_TOK = "<pad>"

def hf_tokenizer(name='gpt2'):
    "Pretrained Hugging Face tokenizer `name`, with `EOS_TOK`, `BOS_TOK` and `PAD_TOK` added as special tokens"
    tok = Tokenizer.from_pretrained(name)
    tok.add_special_tokens([EOS_TOK, BOS_TOK, PAD_TOK])
    return tok

In [35]:
tok = hf_tokenizer()
pad_id = tok.token_to_id(PAD_TOK)
tok.get_vocab_size(), pad_id

(50260, 50259)

## Token cache

A cache stores four files: `flat.npy` for concatenated token IDs, `offsets.npy` for document boundaries, `tokenizer.json` to reload the tokenizer, and `metadata.csv` for document metadata. Raw text is not stored.

In this example, offsets `[0, 2, 3]` describe documents `[1, 2]` and `[3]`. Their metadata rows follow the same order.

In [36]:
def save_token_cache(path, flat, offsets, tok, metadata):
    "Save token IDs, offsets, tokenizer and document metadata to `path`"
    if len(metadata) != len(offsets)-1: raise ValueError('metadata rows must match the number of documents')
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    np.save(path/'flat.npy', flat)
    np.save(path/'offsets.npy', offsets)
    metadata.to_csv(path/'metadata.csv', index=False)
    tok.save(str(path/'tokenizer.json'))

def load_token_cache(path):
    "Load token IDs, offsets, tokenizer and document metadata from `path`"
    path = Path(path)
    flat, offsets = np.load(path/'flat.npy'), np.load(path/'offsets.npy')
    tok = Tokenizer.from_file(str(path/'tokenizer.json'))
    metadata = pd.read_csv(path/'metadata.csv', dtype=str)
    return flat, offsets, tok, metadata

In [37]:
cache_flat = np.array([1, 2, 3], dtype=np.uint16)
cache_offsets = np.array([0, 2, 3], dtype=np.int64)
with tempfile.TemporaryDirectory() as d:
    metadata = pd.DataFrame(dict(path=['train/pos/1.txt', 'test/neg/2.txt'], label=['pos', 'neg'], split=['train', 'test']))
    save_token_cache(d, cache_flat, cache_offsets, tok, metadata)
    flat2, offsets2, tok2, metadata2 = load_token_cache(d)
test_eq(flat2, cache_flat)
test_eq(offsets2, cache_offsets)
test_eq(metadata2.to_dict('list'), metadata.to_dict('list'))
metadata2

,path,label,split
0,train/pos/1.txt,pos,train
1,test/neg/2.txt,neg,test


## Encoding texts

Documents have different lengths. `tokenize_texts` packs their token IDs into `flat` and records the boundaries in `offsets`. Document `i` occupies `flat[offsets[i]:offsets[i+1]]`.

In [38]:
def id_dtype(tok):
    "The smallest type that holds every ID of `tok`"
    return np.uint16 if tok.get_vocab_size(with_added_tokens=True) <= 2**16 else np.uint32

In [39]:
def tokenize_texts(
    items, # Texts, or the items `get_texts` extracts them from
    tok, # HF `Tokenizer`
    get_texts=None, # Column of texts from Dataset, or function from `items` to the texts
    bs=1024, 
):
    "Encode texts into flat token IDs and document offsets, where document `i` is `flat[offsets[i]:offsets[i+1]]`"
    if isinstance(get_texts, str): get_texts = itemgetter(get_texts)
    texts = items if get_texts is None else get_texts(items)
    texts = list(texts)
    dtype = id_dtype(tok)
    chunks = [np.asarray(e.ids, dtype=dtype) for i in progress_bar(range(0, len(texts), bs)) for e in tok.encode_batch_fast(texts[i:i+bs])]
    flat = np.concatenate(chunks) if chunks else np.empty(0, dtype=dtype)
    offsets = np.cumsum([0] + [len(c) for c in chunks], dtype=np.int64)
    return flat, offsets

 what is the optimal batch size to choose?

In [40]:
texts = ["The quick brown fox", "jumps over the lazy dog"]
flat, offsets = tokenize_texts(texts, tok)
test_eq(offsets[-1], len(flat))
flat, offsets

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

(array([  464,  2068,  7586, 21831,    73,  8142,   625,   262, 16931,
         3290], dtype=uint16),
 array([ 0,  4, 10]))

Slice between two adjacent offsets to recover a document, then decode it with the same tokenizer.

In [41]:
first_doc = flat[offsets[0]:offsets[1]]
tok.decode(first_doc.tolist())

'The quick brown fox'

For tabular input, `get_texts` names the text column. 


In [42]:
df = pd.DataFrame({'id': range(len(texts)), 'text': texts})
flat_df, offsets_df = tokenize_texts(df, tok, get_texts='text')
test_eq(flat_df, flat)
test_eq(offsets_df, offsets)


 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

`TokenDocs` wraps `flat` and `offsets`. Indexing retrieves a document view without creating all the views upfront. Subsets retain document indices. The `iloc` property lets fastai index this collection directly without converting it to a list.

In [43]:
class TokenDocs:
    "Document views backed by flat token IDs and offsets"
    def __init__(
        self,
        flat, # Concatenated token IDs
        offsets, # Document boundaries in flat
        idxs=None # Document indices for a subset; None includes every document
    ):
        self.flat,self.offsets = flat,offsets
        self.idxs = np.arange(len(offsets)-1) if idxs is None else idxs

    def __len__(self): return len(self.idxs)
    def __repr__(self): return f'TokenDocs({len(self)} documents)'
    @property
    def iloc(self): return self

    def __getitem__(self, i):
        i = self.idxs[i]
        if not is_indexer(i): return type(self)(self.flat, self.offsets, i)
        return self.flat[self.offsets[i]:self.offsets[i+1]]

In [44]:
docs = TokenDocs(flat, offsets)
test_eq([tok.decode(d.tolist()) for d in docs], texts)
test_is(docs.flat, flat)
test_eq(docs[[1, 0]][0], docs[-1])
test_eq(np.shares_memory(docs[0], flat), True)
tok.decode(docs[1].tolist())

'jumps over the lazy dog'

## Hugging Face datasets

We’ll prepare datasets for training by tokenizing their documents once and preserving the matching labels and source splits. The resulting cache can supply both language-model and classification loaders.

We’ll start with local CSV files and SimpleWiki, where each row is one document. WikiText shows how to prepare data when an article spans several rows.


In [45]:
def get_dataset_data(
    name, # Hugging Face Hub dataset name or local dataset directory
    get_data=None, # Optional callable returning (texts, metadata DataFrame) for each split
    subset=None,
    split=None, # All splits if None, a split name or list of splits
    text_col=None, # Text column; required when get_data is not supplied
    metadata_cols=(), # Columns to cache alongside the source split
):
    "Load selected dataset splits and combine their prepared texts and metadata"
    if get_data is None and text_col is None: raise ValueError('Provide text_col or get_data')
    dsets = load_dataset(name, subset, split=split)
    if split is None: dsets = dsets.values()
    elif isinstance(split, str): dsets = [dsets]
    texts, metadata = [], []
    for ds in dsets:
        if get_data is None:
            split_texts = ds[text_col]
            split_metadata = pd.DataFrame({c: ds[c] for c in metadata_cols}, index=range(len(ds)))
            split_metadata['split'] = str(ds.split)
        else: split_texts, split_metadata = get_data(ds)
        texts.extend(split_texts)
        metadata.append(split_metadata)
    return texts, pd.concat(metadata, ignore_index=True)

In [46]:
@delegates(get_dataset_data)
def tokenize_dataset(
    name, # Hugging Face Hub dataset name or local dataset directory
    subset=None,
    split=None, # All splits if None, or a split name/expression or list of splits
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    cache_dir=None, # Cache directory; defaults to a path keyed by source, selected splits and tokenizer
    bs=1024, # Texts per tokenizer batch
    **kwargs
):
    "Build or load a token cache for a Hugging Face Hub dataset or local dataset directory"
    split_key = 'all' if split is None else str(split)
    if cache_dir is None: cache_dir = fastai_path('data')/'token_cache'/name/(subset or 'default')/split_key/tok
    if not (Path(cache_dir)/'metadata.csv').exists():
        texts, metadata = get_dataset_data(name, subset=subset, split=split, **kwargs)
        tok = hf_tokenizer(tok)
        flat, offsets = tokenize_texts(texts, tok, bs=bs)
        save_token_cache(cache_dir, flat, offsets, tok, metadata)
    return load_token_cache(cache_dir)


These local CSVs contain one review per row. Loading them uses the same column-based interface as a Hub dataset, and each metadata row records its source split.

In [47]:
from contextlib import contextmanager

In [48]:
@contextmanager
def sample_dataset():
    with tempfile.TemporaryDirectory() as d:
        source = Path(d)/'source'
        source.mkdir()
        for split_name in ['train', 'validation', 'test']:
            (source/f'{split_name}.csv').write_text(rf"""text,label
Great {split_name} film,pos
Awful {split_name} film,neg
""")
        yield str(source), Path(d)/'cache'

`split=None` reads all source splits. Pass `split='train'` to select one, or `split=['train', 'validation']` to combine several. The metadata records each document's source split.

In [49]:
with sample_dataset() as (source, _):
    _, all_metadata = get_dataset_data(source, text_col='text')
    test_eq(set(all_metadata.split), {'train', 'validation', 'test'})
    _, train_metadata = get_dataset_data(source, text_col='text', metadata_cols=['label'], split='train')
    test_eq(train_metadata.split.tolist(), ['train', 'train'])
train_metadata

Generating train split: 2 examples [00:00, 2521.37 examples/s]
Generating validation split: 2 examples [00:00, 4457.28 examples/s]
Generating test split: 2 examples [00:00, 5062.53 examples/s]


,label,split
0,pos,train
1,neg,train


Cache the training and validation reviews with their labels

In [50]:
with sample_dataset() as (source, cache_dir):
    _, _, _, review_metadata = tokenize_dataset(source, text_col='text', metadata_cols=['label'],
        split=['train', 'validation'], cache_dir=cache_dir)
    test_eq(review_metadata.split.tolist(), ['train', 'train', 'validation', 'validation'])
    test_eq(review_metadata.label.tolist(), ['pos', 'neg']*2)
review_metadata

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

Generating train split: 2 examples [00:00, 3184.74 examples/s]
Generating validation split: 2 examples [00:00, 4505.16 examples/s]
Generating test split: 2 examples [00:00, 5074.78 examples/s]


,label,split
0,pos,train
1,neg,train
2,pos,validation
3,neg,validation


Supply either a text column or a custom data callable when preparing a dataset.

In [51]:
test_fail(lambda: get_dataset_data('stanfordnlp/imdb'), exc=ValueError, contains='Provide text_col or get_data')

Simple English Wikipedia has one article per row in its `md` column. Pass `text_col='md'` to tokenize those articles. `tokenize_dataset` adds each article's source split to the metadata.

In [52]:
#| eval: false
flat_sw, offsets_sw, _, metadata_sw = tokenize_dataset('answerdotai/simplewiki', subset='articles',
    text_col='md', split='train[:500]')
len(offsets_sw)-1, len(flat_sw)


(500, 752440)

In [53]:
#| eval: false
tok.decode(TokenDocs(flat_sw, offsets_sw)[0].tolist())[:80]


'**April** (Apr.) is the fourth month of the year in the Julian and Gregorian cal'

WikiText-103 spreads each article across several rows, with headings marking article boundaries. Here a custom `get_data` callable joins those rows and returns `(texts, metadata)`, with one metadata row per assembled article. It runs separately on each source split to keep articles within their split.

In [54]:
_wiki_heading = re.compile(r'^ = [^=].* = \n$')

def get_wiki103_data(ds):
    "Assemble WikiText-103 articles and their metadata from source lines"
    docs, cur = [], []
    for text in ds['text']:
        if _wiki_heading.match(text):
            if cur: docs.append(''.join(cur))
            cur = []
        cur.append(text)
    if cur: docs.append(''.join(cur))
    texts = [d for d in docs if d.strip()]
    return texts, pd.DataFrame(dict(split=[str(ds.split)]*len(texts)))

In [55]:
#| eval: false
flat_wiki, offsets_wiki, tok_wiki, metadata_wiki = tokenize_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, split='validation')
len(offsets_wiki)-1, len(flat_wiki)


(60, 247289)

In [56]:
#| eval: false
sorted(p.name for p in (fastai_path('data')/'token_cache/Salesforce/wikitext/wikitext-103-raw-v1/validation/gpt2').iterdir())

['flat.npy', 'metadata.csv', 'offsets.npy', 'tokenizer.json']

In [57]:
#| eval: false
tok_wiki.decode(TokenDocs(flat_wiki, offsets_wiki)[0].tolist())[:80]


' = Homarus gammarus = \n Homarus gammarus , known as the European lobster or comm'

## Text files

For the standard `split/label/file` layout, directory names supply the label and source split. The relative path identifies the document. `get_folder_data` sorts the files and keeps metadata in that order.

For other layouts, `get_metadata` receives the sorted relative paths and returns extra metadata columns with one row per file. The paths are added automatically.

In [58]:
def get_folder_data(
    path, # Folder of text files
    get_metadata=None # Callable returning metadata columns for nonstandard folder layouts
):
    "List sorted text files under `path` and prepare their metadata"
    path = Path(path)
    files = get_text_files(path).sorted()
    rel_files = [f.relative_to(path) for f in files]
    paths = [str(f) for f in rel_files]
    if get_metadata is None:
        metadata = pd.DataFrame(dict(
            path=paths,
            label=[f.parent.name if len(f.parts) >= 2 else None for f in rel_files],
            split=[f.parents[1].name if len(f.parts) >= 3 else None for f in rel_files]))
    else:
        metadata = get_metadata(rel_files)
        if len(metadata) != len(files): raise ValueError('get_metadata must return one row per file')
        metadata = pd.concat([pd.DataFrame(dict(path=paths)), metadata.reset_index(drop=True)], axis=1)
    return files, metadata

In [59]:
@delegates(tokenize_texts, but=['get_texts'])
def tokenize_folder(
    path, # Folder of text files
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    get_metadata=None, # Callable returning metadata columns for nonstandard folder layouts
    cache_dir=None, # Cache directory; `fastai_path('data')/'token_cache'/path.name/tok` if None
    **kwargs
):
    "Cache text files under `path` with paths and metadata"
    path = Path(path)
    files, metadata = get_folder_data(path, get_metadata)
    if cache_dir is None: cache_dir = fastai_path('data')/'token_cache'/path.name/tok
    cache_meta = Path(cache_dir)/'metadata.csv'
    if cache_meta.exists():
        flat, offsets, tokenizer, cached_metadata = load_token_cache(cache_dir)
        if 'path' in cached_metadata and metadata['path'].tolist() == cached_metadata['path'].tolist():
            docs = TokenDocs(flat, offsets)
            if len(docs) != len(files): raise ValueError(f'{cache_dir} holds {len(docs)} documents for {len(files)} files: delete it to rebuild')
            return files, docs, tokenizer, cached_metadata
    tokenizer = hf_tokenizer(tok)
    flat, offsets = tokenize_texts(files, tokenizer,
        get_texts=lambda fs: [f.read_text(encoding='utf-8') for f in fs], **kwargs)
    save_token_cache(cache_dir, flat, offsets, tokenizer, metadata)
    return files, TokenDocs(flat, offsets), tokenizer, pd.read_csv(cache_meta, dtype=str)

`tokenize_folder` returns files, documents, tokenizer and metadata in matching order. Inspecting a file's metadata alongside its decoded tokens makes that relationship visible.

Matching cached paths reuse the saved tokens and metadata; changed paths rebuild them. To change labels or splits for unchanged paths, delete the cache or choose a new `cache_dir`. Rebuild a cache that lacks metadata columns needed by a loader.

In [60]:
#| eval: false
path = untar_data(URLs.IMDB)
files, imdb_docs, tok_imdb, imdb_metadata = tokenize_folder(path)
len(files), files[0].relative_to(path), imdb_metadata.iloc[0], tok_imdb.decode(imdb_docs[0][:30].tolist())

(100002,
 Path('test/neg/0_2.txt'),
 path     test/neg/0_2.txt
 label                 neg
 split                test
 Name: 0, dtype: str,
 'Once again Mr. Costner has dragged out a movie for far longer than necessary. Aside from the terrific sea rescue sequences, of which there are very')

## Transforms

Cached token IDs still need boundary tokens and a tensor type for training. `HfTokenizer` wraps each document in a `TensorText` with `<bos>` and `<eos>`, leaving the cached arrays unchanged. It also accepts raw strings.

In [61]:
BOS_TOK, EOS_TOK, PAD_TOK

('<bos>', '<eos>', '<pad>')

In [62]:
class HfTokenizer(Transform):
    "Encode texts or cached token IDs with `tok`, adding `BOS_TOK` and `EOS_TOK`, and decode them back to text"
    input_types = (str, np.ndarray)
    def __init__(self, tok):
        self.tok = tok
        self.bos,self.eos,self.pad_id = (tok.token_to_id(t) for t in (BOS_TOK, EOS_TOK, PAD_TOK))
        self.vocab = [tok.id_to_token(i) for i in range(tok.get_vocab_size())]
    def _add_boundaries(self, ids): return TensorText([self.bos] + ids + [self.eos])
    def _len(self, o): return len(o) if isinstance(o, np.ndarray) else len(self.tok.encode(o).ids)
    def get_lengths(self, items):
        if isinstance(items, TokenDocs): return (np.diff(items.offsets)+2)[items.idxs]
        items = L(items)
        if not items: return L()
        return L(self._len(o)+2 for o in items) if isinstance(items[0], (np.ndarray, str)) else None
    def encodes(self, s:str): return self._add_boundaries(self.tok.encode(s).ids)
    def encodes(self, s:np.ndarray): return self._add_boundaries(s.tolist())
    def decodes(self, t:TensorText): return TitledStr(self.tok.decode(t.tolist(), skip_special_tokens=False))

TensorDocs vs TensorText

The boundaries add two tokens per document. `get_lengths` reports these lengths to `LMDataLoader` and `SortedDL`

In [63]:
tfm = HfTokenizer(tok)
t = tfm(docs[0])
test_eq(tfm.get_lengths(texts), tfm.get_lengths(docs))
test_eq(tfm.get_lengths(docs[[1, 0]]), tfm.get_lengths(texts)[[1, 0]])
test_eq(tfm.get_lengths([]), [])
t, tfm.decode(t), tfm.get_lengths(docs)

(TensorText([50258,   464,  2068,  7586, 21831, 50257]),
 '<bos>The quick brown fox<eos>',
 array([6, 8]))

A raw string and its cached IDs produce the same `TensorText`. Decoding keeps the boundary tokens visible.

In [64]:
test_eq(tfm(texts[0]), t)
tfm(texts[1]), tfm.decode(tfm(texts[1]))

(TensorText([50258,    73,  8142,   625,   262, 16931,  3290, 50257]),
 '<bos>jumps over the lazy dog<eos>')

Place `Reverse` after `HfTokenizer` to train on reversed sequences. It reverses on decoding too, restoring the original text for display. The same loaders and learners can use either direction.

In [65]:
class Reverse(Transform):
    def encodes(self, o): return o.flip(0)
    def decodes(self, o): return o.flip(0)


In [66]:
pipe = Pipeline([tfm, Reverse()])
test_eq(pipe(docs[0]), t.flip(0))
pipe.decode(pipe(docs[0]))


'<bos>The quick brown fox<eos>'

## DataLoaders

`HfTextBlock` uses `LMDataLoader` for language modeling. For classification, it uses `SortedDL` and pads documents with the tokenizer's `pad_id`.

In [71]:
class HfTextBlock(TransformBlock):
    "A `TransformBlock` for documents that `tok_tfm` encodes straight to token ids"
    def __init__(self, tok_tfm, is_lm=False, seq_len=72, backwards=False):
        type_tfms = [tok_tfm] + ([Reverse()] if backwards else [])
        super().__init__(type_tfms=type_tfms, dl_type=LMDataLoader if is_lm else SortedDL,
                         dls_kwargs={'seq_len': seq_len} if is_lm else {'before_batch': Pad_Chunk(seq_len=seq_len, pad_idx=tok_tfm.pad_id)})

A language model predicts the next token in the stream. In this batch, the targets in `y` are shifted one position ahead of the inputs in `x`.

In [72]:
blk = HfTextBlock(tfm, is_lm=True, seq_len=3)
dls = DataBlock(blocks=blk, splitter=RandomSplitter(0.5, seed=42)).dataloaders(docs, bs=1)
x, y = dls.one_batch()
test_eq(x[:, 1:], y[:, :-1])
x, y

(LMTensorText([[50258,    73,  8142]], device='mps:0'),
 TensorText([[  73, 8142,  625]], device='mps:0'))

### Selecting documents

Source splits describe where documents came from. Loader splits select document positions for training and validation. `get_metadata_splits` uses the saved split names, or `valid_pct` to make a new random split from all selected rows. 

In [76]:
def get_metadata_splits(
    metadata, # One metadata row per cached document
    train='train', # Training split name
    valid='validation', # Validation split name
    valid_pct=None, # Random validation fraction; None uses named source splits
    seed=None, # Random split seed
    folders=None # Top-level source folders to include; valid_pct splits all their rows
):
    "Training and validation row positions from cached metadata"
    rows = L(range_of(metadata))
    if folders is not None:
        folders = L(folders)
        rows = L(mask2idxs(metadata['path'].map(lambda p: Path(p).parts[0] in folders)))
        metadata = metadata.iloc[rows]
    if valid_pct is not None:
        train_idx, valid_idx = RandomSplitter(valid_pct, seed=seed)(rows)
        return rows[train_idx], rows[valid_idx]
    return [rows[mask2idxs(metadata['split'] == name)] for name in (train, valid)]

In [77]:
split_metadata = pd.DataFrame(dict(split=['train', 'train', 'validation', 'test']))
split_indices = get_metadata_splits(split_metadata)
test_eq(split_indices, ([0, 1], [2]))
test_eq(get_metadata_splits(split_metadata.iloc[:0]), ([], []))
split_indices

[[0, 1], [2]]

### Loader factories

`TextDataLoadersNew` is the replacement under development for `TextDataLoaders`. `from_docs` builds loaders from `TokenDocs`, with labels supplied separately. `from_dataset` and `from_folder` obtain those documents and aligned metadata through their token caches. Both factories accept the loader options of `from_docs`.

In [64]:
class TextDataLoadersNew(DataLoaders):
    "Wrapper around several `DataLoader`s with factory methods for texts encoded by `HfTokenizer`"
    @classmethod
    @delegates(Datasets.dataloaders)
    def from_docs(
        cls,
        docs, # TokenDocs backed by flat token IDs and offsets
        tok, # Tokenizer used to encode the documents
        splits, # Training and validation document indices
        labels=None, # One label per document; None for a language model
        seq_len=72, # Language model sequence length or classifier padding chunk size
        backwards=False, # Reverse each tokenized document
        bs=64, # DataLoader batch size
        **kwargs
    ):
        r"Build language model or classification loaders from tokenized documents"
        is_lm = labels is None
        text = HfTextBlock(HfTokenizer(tok), is_lm=is_lm, seq_len=seq_len, backwards=backwards)
        tls = [TfmdLists(docs, text.type_tfms, splits=splits)]
        if not is_lm: tls.append(TfmdLists(labels, CategoryBlock().type_tfms, splits=splits))
        dsets = Datasets(tls=tls, dl_type=text.dl_type)
        kwargs = {**text.dls_kwargs, 'after_item': text.item_tfms, 'after_batch': text.batch_tfms, **kwargs}
        return dsets.dataloaders(bs=bs, **kwargs)

    @classmethod
    @delegates(from_docs, but=['labels'])
    def from_dataset(
        cls,
        name, # Hub dataset name or local dataset directory
        train='train', # Source split used for training
        valid='validation', # Source split used for validation
        valid_pct=None, # Random validation fraction of selected rows; None uses source splits
        seed=None, # Random split seed
        tok='gpt2', # Tokenizer name for hf_tokenizer
        cache_dir=None, # Token cache directory
        label_col=None, # Label column in cached metadata; None for a language model
        metadata_cols=(), # Additional source columns to cache; label_col is included automatically
        subset=None, # Dataset configuration name
        text_col=None, # Text column; required when get_data is not supplied
        get_data=None, # Callable returning texts and metadata for each source split
        **kwargs
    ):
        "Build language model or classification loaders using cached document metadata"
        metadata_cols = (L(metadata_cols) + L(label_col)).unique()
        split = [train, valid] if valid_pct is None else None
        flat, offsets, tok, metadata = tokenize_dataset(name, subset=subset, split=split, tok=tok,
            cache_dir=cache_dir, text_col=text_col, get_data=get_data, metadata_cols=metadata_cols)
        docs = TokenDocs(flat, offsets)
        splits = get_metadata_splits(metadata, train=train, valid=valid, valid_pct=valid_pct, seed=seed)
        labels = None if label_col is None else metadata[label_col]
        return cls.from_docs(docs, tok, splits, labels=labels, **kwargs)

    @classmethod
    @delegates(from_docs, but=['labels'])
    def from_folder(
        cls,
        path, # Folder of text files
        train='train', # Training split name in cached metadata
        valid='valid', # Validation split name in cached metadata
        valid_pct=None, # Random validation fraction; None uses source splits
        seed=None, # Random split seed
        is_lm=False, # Build a language model instead of a classifier
        tok='gpt2', # Tokenizer name for hf_tokenizer
        folders=None, # Source folders to include; with valid_pct, combine their rows before splitting
        cache_dir=None, # Token cache directory
        label_col='label', # Cached label column for classification
        get_metadata=None, # Callable returning metadata columns for nonstandard folder layouts
        **kwargs
    ):
        "Build language model or classification loaders from a folder's cached metadata"
        _, docs, tok, metadata = tokenize_folder(path, tok, get_metadata=get_metadata, cache_dir=cache_dir)
        splits = get_metadata_splits(metadata, train=train, valid=valid, valid_pct=valid_pct, seed=seed, folders=folders)
        labels = None if is_lm else metadata[label_col]
        return cls.from_docs(docs, tok, splits, labels=labels, path=path, **kwargs)


For quick experiments, use `tokenize_texts` and `TokenDocs` to prepare documents in memory, then build loaders with `from_docs`. No cache files are written. Pass `labels` for classification or omit them for language modeling.

In [65]:
memory_dls = TextDataLoadersNew.from_docs(docs, tok, splits=([0], [1]), bs=1, seq_len=3)
test_is(memory_dls.train_ds.items.flat, flat)
memory_dls.one_batch()

(LMTensorText([[50258,   464,  2068]], device='mps:0'),
 TensorText([[ 464, 2068, 7586]], device='mps:0'))

`from_dataset` builds a language model unless `label_col` is supplied. For classification, it includes that column in the cached metadata automatically; a custom `get_data` callable must provide it.



In [66]:
with sample_dataset() as (source, cache_dir):
    dls = TextDataLoadersNew.from_dataset(source, text_col='text', label_col='label',
        cache_dir=cache_dir, bs=2, val_bs=1, device='cpu')
    tokenizer = Tokenizer.from_file(str(cache_dir/'tokenizer.json'))
train_items = [(tokenizer.decode(doc.tolist()), dls.vocab[1][label]) for doc, label in dls.train_ds]
valid_items = [(tokenizer.decode(doc.tolist()), dls.vocab[1][label]) for doc, label in dls.valid_ds]
test_eq(train_items, [('Great train film', 'pos'), ('Awful train film', 'neg')])
test_eq(valid_items, [('Great validation film', 'pos'), ('Awful validation film', 'neg')])
test_is(dls.train_ds.items.flat, dls.valid_ds.items.flat)
train_items, valid_items

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

Generating train split: 2 examples [00:00, 2558.28 examples/s]
Generating validation split: 2 examples [00:00, 4236.67 examples/s]
Generating test split: 2 examples [00:00, 4865.78 examples/s]


([('Great train film', 'pos'), ('Awful train film', 'neg')],
 [('Great validation film', 'pos'), ('Awful validation film', 'neg')])

For classification, `x` contains padded token sequences and `y` contains category indices. The vocabulary maps those indices back to labels.

In [67]:
x, y = dls.one_batch()
test_eq(dls.vocab[1], ['neg', 'pos'])
dls.vocab[1], x, y

(['neg', 'pos'],
 TensorText([[50258, 23155,   913,  4512,  2646, 50257],
             [50258, 13681,  4512,  2646, 50257, 50259]]),
 TensorCategory([0, 1]))

With `valid_pct`, `from_dataset` pools every source split before making a new random split. This includes unsupervised documents and any test data. Here all six local reviews participate, including the two from the test source.

In [ ]:
with sample_dataset() as (source, cache_dir):
    all_dls = TextDataLoadersNew.from_dataset(source, text_col='text', valid_pct=0.2,
        seed=42, cache_dir=cache_dir, bs=1, seq_len=3)
    source_counts = pd.read_csv(cache_dir/'metadata.csv')['split'].value_counts()
test_eq(len(all_dls.train_ds) + len(all_dls.valid_ds), source_counts.sum())
dict(sources=source_counts.to_dict(), train=len(all_dls.train_ds), valid=len(all_dls.valid_ds))

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

Generating train split: 2 examples [00:00, 969.45 examples/s]
Generating validation split: 2 examples [00:00, 4025.24 examples/s]
Generating test split: 2 examples [00:00, 4899.89 examples/s]


{'sources': {'train': 2, 'validation': 2, 'test': 2}, 'train': 5, 'valid': 1}

For language modeling, leave `label_col` unset. This WikiText example uses the article preparation callable from earlier. `train='train[:1%]'` selects a slice, with that exact expression recorded as its split name. The other source is the named validation split.

In [68]:
#| eval: false
dls = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:1%]', bs=8)
dls.show_batch(max_n=2)

<bos> = Crazy in Love = 
 " Crazy in Love " is a song from American singer Beyoncé 's debut solo album Dangerously in Love ( 2003 ) . Beyoncé wrote the song with Rich Harrison , Jay Z , and Eugene Record , and produced it with Harrison . " Crazy in Love " is an R & B and pop love song that incorporates
 . 
 In 1904 , George Clinch published a note on the Medway Megaliths in the Royal Anthropological Institute 's journal , Man , in which he referred to the Coldrum Stones as " at once the most remarkable and the least known of the whole series . " Suggesting that its design indicates that it was built during " a late date
 = Crazy in Love = 
 " Crazy in Love " is a song from American singer Beyoncé 's debut solo album Dangerously in Love ( 2003 ) . Beyoncé wrote the song with Rich Harrison , Jay Z , and Eugene Record , and produced it with Harrison . " Crazy in Love " is an R & B and pop love song that incorporates elements
 
 In 1904 , George Clinch published a note on the Medway Megali

For classification, use named labeled sources. Here `valid='test'` uses IMDb's test reviews for validation; other source splits are left out.

In [69]:
#| eval: false
dls = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', label_col='label',
    valid='test', bs=8)
dls.show_batch(max_n=2)


,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",1
1,"<bos>That word 'True' in this film's title got my alarm bells ringing. They rang louder when a title card referred to America's Civil War as the 'War Between the States' (the circumlocution preferred by die-hard southerners). Jesse James -- thief, slave-holder and murderer -- is described as a quiet, gentle farm boy.<br /><br />How dishonest is this movie? There is NO mention of slavery, far less of the documented fact that Jesse James's poor widdered mother owned slaves before the war, and that Jesse and his brother Frank actively fought to preserve slavery. According to this movie, all those Civil War soldiers were really fighting to decide whether Missouri is a northern state or a southern state ... that's ALL. (Missouri: It's a candy mint! It's a breath mint!) Black people are entirely absent from this movie, except for two glimpses of a pair of beggars, one of whom wears",0


### Folder loaders

For IMDb, the `train` and `test` folders provide split names and `pos` and `neg` provide labels. `from_folder` reads these from cached metadata. Set `label_col` for another label column or pass `get_metadata` for a custom layout. Here `valid='test'` uses the test reviews for validation.


A small local folder uses the same `split/label/file` layout as IMDb. Both loader types keep their documents in the shared token array.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    folder = Path(d)
    for split in ('train', 'valid'):
        for label, text in [('pos', 'Great film'), ('neg', 'Awful film')]:
            review = folder/split/label/'review.txt'
            review.parent.mkdir(parents=True)
            review.write_text(text)
    folder_dls = TextDataLoadersNew.from_folder(folder, cache_dir=folder/'cache', bs=2, device='cpu')
    folder_lm = TextDataLoadersNew.from_folder(folder, cache_dir=folder/'cache', is_lm=True,
        backwards=True, bs=1, seq_len=3, device='cpu')
test_eq([folder_dls.vocab[1][label] for _, label in folder_dls.valid_ds], ['neg', 'pos'])
test_is(folder_dls.train_ds.items.flat, folder_dls.valid_ds.items.flat)
folder_dls.one_batch(), folder_lm.one_batch()

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

((TensorText([[50258, 23155,   913,  2646, 50257],
              [50258, 13681,  2646, 50257, 50259]]),
  TensorCategory([0, 1])),
 (LMTensorText([[50257,  2646,   913]]), TensorText([[ 2646,   913, 23155]])))

In [70]:
#| eval: false
dls = TextDataLoadersNew.from_folder(path, valid='test', bs=8)
dls.show_batch(max_n=2)

,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",pos
1,"<bos>Shlock-merchant Leo Fulci takes a change of pace by making a trashy, barely coherent sword and sorcery fantasy movie instead of his usual trashy, barely coherent horror. <br /><br />A wimpy Orlando Bloom type called Ilias, from some society vaguely resembling Ancient Greece travels across the ocean to caveman territory on some vaguely defined quest to battle evil, where he joins up with a animal loving hunter to battle the wolf-man and mutant minions of a vampiric topless evil sorceress. Wackiness ensues. The sorceress, is oppressing the local cavemen and wants the magic bow for herself. She sends various minions, each weirder than the last, after our heroes who win through in the end, striking a blow for oppressed cavemen everywhere. This movie contains a steady stream of WTF? elements and moments.<br /><br />For some reason the entire movie is shot in soft focus and the picture is further blurred",neg


For language modeling, the random split below pools all cached documents, including unsupervised reviews. Pass `folders` to restrict that pool.

In [71]:
#| eval: false
dls_lm = TextDataLoadersNew.from_folder(path, is_lm=True, valid_pct=0.1, bs=64)
len(dls_lm.train_ds), len(dls_lm.valid_ds), len(dls_lm.vocab)

(90002, 10000, 50260)

## Export -

In [72]:
#| hide
from nbdev import nbdev_export
nbdev_export()

## TokenDocs and fastcore collection handling

`from_docs` passes `TokenDocs` to `TfmdLists`, which initializes fastcore's `L` with `use_list=None`. In that mode, `L` skips `listify` only when `is_array(items)` is true:

```python
return hasattr(x, '__array__') or hasattr(x, 'iloc')
```

`TokenDocs.iloc` returns `self`. This keeps the original collection intact and lets `L._get` select documents through `self.items.iloc[list(i)]`. The resulting subset shares the original token array and offsets.

Without `iloc`, the current class is wrapped as one item, `[docs]`. Python can iterate through its `__getitem__`, but fastcore's `is_iter` checks `collections.abc.Iterable`, which does not recognize this class without `__iter__`. Adding `__iter__` without `iloc` would make `listify` build a list of document views instead.

The relevant implementations are [`TfmdLists.__init__`](/Users/hoa/aai-ws/fastai/fastai/data/core.py:344), [`L.__init__` and `L._get`](/Users/hoa/aai-ws/fastcore/fastcore/foundation.py:142), [`is_array` and `listify`](/Users/hoa/aai-ws/fastcore/fastcore/basics.py:115), and [`is_iter`](/Users/hoa/aai-ws/fastcore/fastcore/imports.py:20).

In [32]:
demo = TokenDocs(np.array([10, 11, 20, 21, 22, 30]), np.array([0, 2, 5, 6]))
demo[0], demo[1], demo[2]

(array([10, 11]), array([20, 21, 22]), array([30]))

For this collection, `use_list` has three distinct behaviors:

| Setting | Result |
|---|---|
| `False` (the default) | Wrap the entire collection as one item, `[demo]`. |
| `None` | Keep `demo` as the backing collection because it has `iloc`. |
| `True` | Create a list containing a NumPy view for every document. |

Materializing the views does not copy the underlying token values. The example below checks that indexing and selecting a subset preserve the shared storage.

In [ ]:
wrapped = L(demo, use_list=None)
materialized = L(demo, use_list=True)
test_is(wrapped.items, demo)
test_eq(wrapped[1], demo[1])
test_is(wrapped[[2, 0]].items.flat, demo.flat)
dict(default=L(demo), preserved=wrapped, materialized=materialized)

{'default': [TokenDocs(3 documents)],
 'preserved': TokenDocs(3 documents),
 'materialized': [array([10, 11]), array([20, 21, 22]), array([30])]}

There is still an iteration mismatch. `list(demo)` works through Python's `__getitem__` fallback, but `list(wrapped)` currently raises `AttributeError` because [`L.__iter__`](/Users/hoa/aai-ws/fastcore/fastcore/foundation.py:177) assumes an object with `iloc` also has `itertuples`. The loader examples passed without `itertuples` because they did not exercise that branch. Removing `iloc` broke the language-model loader example.

Adding `__array__` solely to pass `is_array` would introduce another problem: `L._get` requests array conversion before selecting indices. A generic interface for preserving and indexing lazy collections in fastcore remains an open design question. We have not benchmarked the memory or startup savings of `TokenDocs`.